# Python Practice — Day 5
## Handling Missing Data (NaN)

So far our student data has been clean. Real datasets almost always have gaps — a missing score, a blank name, etc. Pandas represents missing values as `NaN` (Not a Number).

**Tools we'll use today:**
- `df.isna()` — True/False mask showing where values are missing
- `df["col"].isna().sum()` — count how many values are missing in a column
- `df.dropna()` — remove rows containing any missing value
- `df.fillna(value)` — replace missing values with something (a number, a string, a column mean, etc.)

We'll keep using our familiar student name/score DataFrame, but this time with some gaps in it.

### Setup
Run this cell first — it builds a small DataFrame with a couple of missing scores and one missing name, just like a messy real-world file might look.

In [3]:
import pandas as pd
import numpy as np

data = {
    "name": ["Alice", "Bob", "Charlie", None, "Evan", "Farah"],
    "score": [85, np.nan, 78, 92, np.nan, 60]
}
df = pd.DataFrame(data)
df


,name,score
0,Alice,85.0
1,Bob,NaN
2,Charlie,78.0
3,None,92.0
4,Evan,NaN
5,Farah,60.0


---
## Exercise 1: Count missing values

Write a function `count_missing_scores(df)` that returns **how many** rows have a missing `score`.

Think about it before coding: which tool from the list above tells you *how many* values are missing in one column?

In [4]:
def count_missing_scores(df):
    """Return the number of rows where score is missing (NaN)."""
    return df["score"].isna().sum()
    


In [5]:
# Test cell — expected output: 2
print(count_missing_scores(df))


2


---
## Exercise 2: Drop rows with missing scores

Write a function `drop_missing_scores(df)` that returns a **new** DataFrame with any row that has a missing `score` removed. Rows with a missing `name` but a valid `score` should stay.

Hint: `dropna()` by default drops a row if **any** column has a NaN — you'll need to tell it to only look at the `score` column, using the `subset` parameter, e.g. `df.dropna(subset=["score"])`.

In [6]:
def drop_missing_scores(df):
    """Return a new DataFrame with rows that have a missing score removed."""
    return df.dropna(subset = ["score"]) 
    pass


In [7]:
# Test cell — expect 4 rows remaining (Bob and Evan dropped, since their scores are NaN)
result = drop_missing_scores(df)
print(result)

print(len(result))


      name  score
0    Alice   85.0
2  Charlie   78.0
3     None   92.0
5    Farah   60.0
4


---
## Exercise 3: Fill missing scores

Instead of dropping rows, sometimes you want to **fill in** a reasonable placeholder value.

Write a function `fill_missing_scores(df)` that returns a new DataFrame where any missing `score` is replaced with the **average (mean) of the existing scores** — rounded to 1 decimal place.

Hint: `df["score"].mean()` ignores NaN automatically. `fillna(value)` replaces NaNs with `value`.

In [8]:
def fill_missing_scores(df):
    """Return a new DataFrame with missing scores filled in with the mean score (rounded to 1 dp)."""

    average = round(df["score"].mean(),1) 

    df = df.copy() 

    df["score"] = df["score"].fillna(average)  

    return df  


In [9]:
# Test cell — Bob and Evan should now have the same filled-in value (the rounded mean of 85, 78, 92, 60)
filled = fill_missing_scores(df)
print(filled)


      name  score
0    Alice   85.0
1      Bob   78.8
2  Charlie   78.0
3     None   92.0
4     Evan   78.8
5    Farah   60.0


---
## Exercise 4: Mixed review — clean, grade, and sort

This one combines today's new concept with earlier ones (filtering, `.apply()`, `sort_values()`).

Write a function `clean_and_grade(df)` that:
1. Drops any row with a missing `name` **or** missing `score` (both columns this time — no `subset`)
2. Adds a `grade` column using your existing `grade_by_score` logic (A ≥ 90, B ≥ 80, C ≥ 70, else F)
3. Sorts the result by `score`, descending
4. Returns the cleaned, graded, sorted DataFrame

In [10]:
def grade_by_score(score):
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    else:
        return "F"


def clean_and_grade(df):
    """Drop rows with any missing name/score, add a grade column, sort by score descending."""
    df = df.copy()
    # df = df[(df["name"].notna()) & (df["score"].notna())] 
    df = df.dropna(subset = ["name", "score"])
    df["grade"] = df["score"].apply(grade_by_score)

    return df.sort_values( by = "score", ascending = False)



In [11]:
# Test cell — expect 4 rows (the row with missing name AND the rows with missing scores are gone),
# sorted highest score first, each with a correct grade letter
print(clean_and_grade(df))


      name  score grade
0    Alice   85.0     B
2  Charlie   78.0     C
5    Farah   60.0     F


---
## Exercise 5 (optional, quick): Where exactly is the missing data?

Write a function `missing_report(df)` that returns a DataFrame showing **only the rows** that have at least one missing value (in any column) — useful for a quick "what needs cleaning" check before you decide whether to drop or fill.

Hint: `df.isna().any(axis=1)` gives a True/False per row — True if that row has any missing value at all.

In [12]:
def missing_report(df):
    """Return only the rows that have at least one missing value in any column."""
    return df[df.isna().any(axis = 1)]
    


In [13]:
# Test cell — expect 3 rows: Bob, the None-name row, and Evan
print(missing_report(df)) 


   name  score
1   Bob    NaN
3  None   92.0
4  Evan    NaN
